# Python 3.8 on Colab

Please note that the following scripts are intended to be used with a specially modified Colab notebook (i.e., this one). For instructions on how to modify an existing notebook to support Python 3.8

In [ ]:
# Download and execute set up script
!wget -O py38.sh https://raw.githubusercontent.com/j3soon/colab-python-version/main/scripts/py38.sh
!bash py38.sh

In [ ]:
import sys
print("User Current Version:-", sys.version)
!python --version

In [ ]:
# Clone Real-ESRGAN and enter the Real-ESRGAN
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
# Set up the environment
!pip install basicsr
!pip install facexlib
!pip install gfpgan
!pip install -r requirements.txt
!python setup.py develop

# **Restart Session**

In [ ]:
import os
from google.colab import files
import shutil

upload_folder = 'upload'
result_folder = 'results'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)
os.mkdir(upload_folder)
os.mkdir(result_folder)

# upload images
uploaded = files.upload()
for filename in uploaded.keys():
  dst_path = os.path.join(upload_folder, filename)
  print(f'move {filename} to {dst_path}')
  shutil.move(filename, dst_path)

In [ ]:
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth -P weights

In [ ]:
%cd Real-ESRGAN

# 3. Install it
!pip install -e .

# 4. Verify the script exists
!ls | grep inference_realesrgan.py

In [ ]:
import fileinput
import sys

file_path = "/usr/local/lib/python3.8/site-packages/basicsr/data/degradations.py"

# Replace the old import with the new one
for line in fileinput.input(file_path, inplace=True):
    if "from torchvision.transforms.functional_tensor import rgb_to_grayscale" in line:
        sys.stdout.write("from torchvision.transforms.functional import rgb_to_grayscale\n")
    else:
        sys.stdout.write(line)

print("✅ Patched degradations.py successfully")


In [ ]:
import torch
torch.cuda.empty_cache()


In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [ ]:
# Anime 6B pass
!python inference_realesrgan.py \
    -n RealESRGAN_x4plus_anime_6B \
    -i /content/upload/57.jpg \
    -o /content/out_anime.png \
    --tile 1024 --tile_pad 50 --ext png -g 0

# General x4 pass
!python inference_realesrgan.py \
    -n RealESRGAN_x4plus \
    -i /content/out_anime.png \
    -o /content/out_final.png \
    --tile 1024 --tile_pad 50 --ext png -g 0